<a href="https://colab.research.google.com/github/gre1wy/Machine_Learning/blob/main/lab4/SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
import time

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [6]:
!pip install imblearn

In [7]:
from imblearn.over_sampling import SMOTE
from collections import Counter

In [8]:
from sklearn.metrics import confusion_matrix, classification_report
from collections import OrderedDict

# Part 1

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fedesoriano/stellar-classification-dataset-sdss17")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'stellar-classification-dataset-sdss17' dataset.
Path to dataset files: /kaggle/input/stellar-classification-dataset-sdss17


In [10]:
csv_file = os.path.join(path, "star_classification.csv")
df = pd.read_csv(csv_file)

In [11]:
df.drop(['obj_ID', 'run_ID', 'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'MJD', 'fiber_ID', 'plate'], axis=1, inplace=True)
col = df.pop("class")
df.insert(0, "class", col)
cols = ['g', 'z', 'u']
df = df[~(df[cols] == -9999).any(axis=1)]

In [12]:
df["class"]=[0 if i == "GALAXY" else 1 if i == "STAR" else 2 for i in df["class"]]

In [13]:
def rem_outliers(df, factor=1.5):
    dataset = df.copy()
    initial_rows = dataset.shape[0]

    for col in dataset.select_dtypes(include='number').columns:
        q1 = dataset[col].quantile(0.25)
        q3 = dataset[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        dataset = dataset[(dataset[col] >= lower) & (dataset[col] <= upper)]

    removed = initial_rows - dataset.shape[0]
    return dataset, removed
df_clean, deleted = rem_outliers(df)
print("Number of outliers deleted are : ", deleted)

Number of outliers deleted are :  9399


In [14]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90600 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   class     90600 non-null  int64  
 1   alpha     90600 non-null  float64
 2   delta     90600 non-null  float64
 3   u         90600 non-null  float64
 4   g         90600 non-null  float64
 5   r         90600 non-null  float64
 6   i         90600 non-null  float64
 7   z         90600 non-null  float64
 8   redshift  90600 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 6.9 MB


In [27]:
df_clean = df_clean[df_clean["class"] != 2]

# Part 2

In [28]:
X = df_clean.drop('class', axis=1)
y = df_clean['class'].values

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=43, stratify=y
)

# X_train, X_val, y_train, y_val = train_test_split(
#     X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
# )

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
# X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [30]:
# sm = SMOTE(random_state=42)
# print('Original dataset shape %s' % Counter(y_train))
# X_train, y_train = sm.fit_resample(X_train, y_train)
# print('Resampled dataset shape %s' % Counter(y_train))

## Test skitlearn

In [31]:
from sklearn.svm import SVC

In [32]:
test = SVC(kernel = "rbf")
test.fit(X_train, y_train)

SVC()

In [33]:
y_predicted = test.predict(X_test)

In [34]:
print(classification_report(y_test, y_predicted))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     11824
           1       0.96      1.00      0.98      4302

    accuracy                           0.99     16126
   macro avg       0.98      0.99      0.98     16126
weighted avg       0.99      0.99      0.99     16126



## SVM

https://bitmask93.github.io/ml-blog/Sequential-Minimal-Optimization-for-Support-Vector-Machines/

In [54]:



class SVM:
    def __init__(self, C=1.0, kernel="linear", tol=1e-3, max_passes=5,
                 sigma=1.0, degree=3, max_cached_rows=50):
        """
        Sequential Minimal Optimization (SMO)-based Support Vector Machine.
        Supports linear, polynomial, and RBF kernels.

        Parameters:
        -----------
        C : float
            Soft-margin penalty parameter.
        kernel : str
            Kernel type: 'linear', 'poly', 'rbf'.
        tol : float
            Numerical tolerance for KKT violation.
        max_passes : int
            Number of consecutive passes without alpha updates before stopping.
        sigma : float
            RBF kernel width parameter.
        degree : int
            Degree for polynomial kernel.
        max_cached_rows : int
            Maximum number of kernel rows stored in LRU cache.
        """
        self.C = C
        self.kernel_type = kernel
        self.tol = tol
        self.max_passes = max_passes
        self.sigma = sigma
        self.degree = degree

        # LRU cache for kernel rows K[i, :]
        self.max_cached_rows = max_cached_rows
        self.kernel_cache = OrderedDict()

    def kernel(self, x1, x2):
        if self.kernel_type == "linear":
            return np.dot(x1, x2)
        elif self.kernel_type == "poly":
            return (1.0 + np.dot(x1, x2)) ** self.degree
        elif self.kernel_type == "rbf":
            return np.exp(-(np.dot(x1, x1) + np.dot(x2, x2) - 2 * np.dot(x1, x2))
                          / (2 * self.sigma ** 2))
        else:
            raise ValueError("Unknown kernel")

    # Returns kernel row K[i, :] (vectorized), using LRU cache.
    # K[i, j] = kernel(X[i], X[j])
    def get_kernel_row(self, i):
        # Return from cache if exists
        if i in self.kernel_cache:
            row = self.kernel_cache.pop(i)
            self.kernel_cache[i] = row
            return row

        Xi = self.X[i]

        if self.kernel_type == "linear":
            row = self.X @ Xi

        elif self.kernel_type == "poly":
            row = (1.0 + self.X @ Xi) ** self.degree

        elif self.kernel_type == "rbf":
            dot = self.X @ Xi
            d2 = self.X_norm + self.X_norm[i] - 2.0 * dot
            row = np.exp(-d2 / (2 * self.sigma ** 2))

        else:
            raise ValueError("Unknown kernel")

        # Maintain LRU policy
        if len(self.kernel_cache) >= self.max_cached_rows:
            self.kernel_cache.popitem(last=False)
        self.kernel_cache[i] = row

        return row

    # SVM output for training sample i: f(x_i)
    def f(self, i):
        K_row = self.get_kernel_row(i)
        return np.sum(self.alphas * self.y * K_row) + self.b


    # Compute box constraints (L, H) for pair of alphas.
    def compute_L_H(self, ai, aj, yi, yj):
        if yi != yj:
            L = max(0.0, aj - ai)
            H = min(self.C, self.C + aj - ai)
        else:
            L = max(0.0, ai + aj - self.C)
            H = min(self.C, ai + aj)
        return L, H

    # Main SMO optimizer
    def fit(self, X, y_raw):
        """
        Train SVM using SMO algorithm.

        Steps:
        1. Convert labels to {-1, +1}
        2. Initialize alpha = 0, b = 0, and error cache E
        3. Iteratively optimize pairs (i, j) of Lagrange multipliers
        4. Update entire error cache vectorized
        5. Store support vectors for fast prediction
        """

        vals = np.unique(y_raw)
        if len(vals) != 2:
            raise ValueError("Only binary classification is supported")

        y = np.where(y_raw == vals[0], -1.0, 1.0)
        self.class_map = vals

        self.X = X
        self.y = y.astype(float)
        self.n = len(y)

        # Precompute norms for RBF
        self.X_norm = np.sum(self.X * self.X, axis=1) if self.kernel_type == "rbf" else None

        self.kernel_cache.clear()

        self.alphas = np.zeros(self.n)
        self.b = 0.0

        self.E = -self.y.copy()

        passes = 0
        examine_all = True

        while passes < self.max_passes:
            num_changed = 0

            # Outer loop strategy: traverse all points, then only non-bound points
            if examine_all:
                indices = range(self.n)
            else:
                indices = np.where((self.alphas > 0) & (self.alphas < self.C))[0]

            for i in indices:
                E_i = self.E[i]
                r_i = self.y[i] * E_i

                # Check KKT violation
                if not ((r_i < -self.tol and self.alphas[i] < self.C) or
                        (r_i > self.tol and self.alphas[i] > 0)):
                    continue

                # Second choice heuristic: pick j maximizing |E_i - E_j|
                non_bound = np.where((self.alphas > 0) & (self.alphas < self.C))[0]
                if len(non_bound) > 1:
                    candidates = non_bound[non_bound != i]
                    if len(candidates) == 0:
                        candidates = np.arange(self.n)
                else:
                    candidates = np.arange(self.n)

                j = candidates[np.argmax(np.abs(self.E[candidates] - E_i))]

                # Fallback to random j
                if i == j:
                    j = np.random.randint(0, self.n)
                    if j == i:
                        continue

                E_j = self.E[j]
                a_i_old = self.alphas[i]
                a_j_old = self.alphas[j]

                L, H = self.compute_L_H(a_i_old, a_j_old, self.y[i], self.y[j])
                if L == H:
                    continue

                # Kernel rows
                Ki = self.get_kernel_row(i)
                Kj = self.get_kernel_row(j)

                Kii = Ki[i]
                Kjj = Kj[j]
                Kij = Ki[j]

                eta = Kii + Kjj - 2.0 * Kij
                if eta <= 0:
                    continue

                # Update alpha_j
                a_j_new = a_j_old + self.y[j] * (E_i - E_j) / eta
                a_j_new_clipped = np.clip(a_j_new, L, H)

                if abs(a_j_new_clipped - a_j_old) < 1e-6:
                    continue

                # Update alpha_i
                a_i_new = a_i_old + self.y[i] * self.y[j] * (a_j_old - a_j_new_clipped)

                # Compute new bias b
                b_old = self.b

                b1 = self.b - E_i \
                     - self.y[i] * (a_i_new - a_i_old) * Kii \
                     - self.y[j] * (a_j_new_clipped - a_j_old) * Kij

                b2 = self.b - E_j \
                     - self.y[i] * (a_i_new - a_i_old) * Kij \
                     - self.y[j] * (a_j_new_clipped - a_j_old) * Kjj

                if 0 < a_i_new < self.C:
                    self.b = b1
                elif 0 < a_j_new_clipped < self.C:
                    self.b = b2
                else:
                    self.b = 0.5 * (b1 + b2)

                # Store alpha changes
                d_ai = a_i_new - a_i_old
                d_aj = a_j_new_clipped - a_j_old

                self.alphas[i] = a_i_new
                self.alphas[j] = a_j_new_clipped

                # Vectorized error-cache update
                delta_b = self.b - b_old
                self.E += self.y[i] * d_ai * Ki + self.y[j] * d_aj * Kj + delta_b

                num_changed += 1

            # SMO loop-switch logic
            if examine_all:
                examine_all = False
            elif num_changed == 0:
                examine_all = True
                passes += 1
            else:
                passes = 0

        # Extract support vectors (alphas > 0)
        sv_mask = self.alphas > 1e-6
        self.sv_X = self.X[sv_mask]
        self.sv_y = self.y[sv_mask]
        self.sv_alpha = self.alphas[sv_mask]

    # Prediction for a batch of test samples

    def project(self, Xtest):
        Xtest = np.asarray(Xtest)

        # Compute kernel matrix between Xtest and SVs
        if self.kernel_type == "linear":
            K = Xtest @ self.sv_X.T

        elif self.kernel_type == "poly":
            K = (1 + Xtest @ self.sv_X.T) ** self.degree

        elif self.kernel_type == "rbf":
            Xtest_norm = np.sum(Xtest * Xtest, axis=1).reshape(-1, 1)
            SV_norm = np.sum(self.sv_X * self.sv_X, axis=1).reshape(1, -1)
            dot = Xtest @ self.sv_X.T
            dist_sq = Xtest_norm + SV_norm - 2 * dot
            K = np.exp(-dist_sq / (2 * self.sigma ** 2))

        else:
            raise ValueError("Unknown kernel")

        return K @ (self.sv_alpha * self.sv_y) + self.b

    # Final class prediction with mapping back to original labels
    def predict(self, Xtest):
        raw = np.sign(self.project(Xtest))
        return np.where(raw == -1, self.class_map[0], self.class_map[1])


In [55]:
test = SVM(kernel = "rbf")

In [56]:
test.fit(X_train, y_train)

In [57]:
predicted = test.predict(X_test)

In [58]:
print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     11824
           1       0.96      1.00      0.98      4302

    accuracy                           0.99     16126
   macro avg       0.98      0.99      0.98     16126
weighted avg       0.99      0.99      0.99     16126

